In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
bau_data = pd.read_csv("../../data/results/BAU_MSW_updated.csv", index_col = 0)
cir_data = pd.read_csv("../../data/results/CIR_MSW.csv", index_col = 0)
rec_data = pd.read_csv("../../data/results/REC_MSW.csv", index_col = 0)

### Estimating ash quantities for cir, rec, bau

In [ ]:
ash_data = pd.concat([bau_data,pd.concat([rec_data,cir_data],ignore_index = True)], ignore_index = True)

In [ ]:
ash_data["SLASH_bottomAshesWasteInc"] = ash_data['INC_T']*0.28
ash_data["SLASH_flyAshesWasteInc"] = ash_data['INC_T']*0.03
#eu_msw_his["SLASH_boilerAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.03
ash_data.drop(['MSW_GEN_T','INC_T'], axis = 'columns', inplace = True)
ash_data.drop(['DIS%','RCY%','INC%'], axis = 'columns', inplace = True)
ash_data.rename(columns= {'TIME':'Year','LOCATION':'Country'},inplace=  True)

### Reformatting into output structure

In [ ]:
columns = ['Waste Stream', 'Country', 'Year', 'Scenario', 'Substance main parent',
           'Stock/Flow ID', 'Value', 'Unit', 'Data Quality', 'Reference', 'Remark 1',
           'Remark 2', 'Remark 3']

In [ ]:
ash_data= pd.melt(ash_data, id_vars=['Country','Year','Scenario'], value_vars = ['SLASH_bottomAshesWasteInc',
                                'SLASH_flyAshesWasteInc'],var_name='Stock/Flow ID', value_name='Value')

In [ ]:
#changing units
ash_data['Unit'] = 't'

In [ ]:
#Adding waste stream
ash_data['Waste Stream'] = 'SLASH'

In [ ]:
#Adding LowKey codes
subs_main_parent = {'SLASH_flyAshesWasteInc':'19 01 13*','SLASH_bottomAshesWasteInc':'19 01 11*'}
                    #'SLASH_boilerAshesWasteInc':'19 01 15*'}
ash_data['Substance main parent'] = ash_data['Stock/Flow ID'].map(subs_main_parent)

Rearrange columns

In [ ]:
#ash_data['Scenario'] = np.where(ash_data['Year'] <= 2021, 'OBS', 'BAU')
ash_data[['Reference']] = np.nan
ash_data[['Remark 2']] = np.nan

In [ ]:
ash_data.loc[(ash_data["Country"]=='GBR')&(ash_data['Year']<=2021),['Remark 1']] = 'Estimated from OECD MSW incineration quantities'
conditions = (((ash_data["Country"]=='DNK')&(ash_data["Year"]==2010)) | ((ash_data["Country"]=='IRL')&(ash_data["Year"]==2013)) | ((ash_data["Country"]=='IRL')&(ash_data["Year"]==2015)) | ((ash_data["Country"]=='ISL')&(ash_data["Year"]==2019)))
ash_data.loc[conditions,['Remark 1']]='Missing data, estimated from Eurostat MSW incineration quantities of neighbouring years '
ash_data.loc[(ash_data["Country"]!='GBR') & (ash_data['Year']<=2021) & ~conditions,['Remark 1']]='Estimated from Eurostat MSW incineration quantities'

Remark for BAU scenario

In [ ]:
conditions = (((ash_data["Country"]=='IRL')&(ash_data["Year"]==2021)) | ((ash_data["Country"]=='GRC')&(ash_data["Year"]==2020))| ((ash_data["Country"]=='GRC')&(ash_data["Year"]==2021)))
ash_data.loc[(ash_data['Scenario']=='BAU') | conditions,['Remark 1']] = 'Estimated from models - Check Reference'
ash_data.loc[(ash_data['Scenario']=='CIR') | (ash_data['Scenario']=='REC'),['Remark 1']] = 'Estimated from models - Check Reference'

In [ ]:
ash_data[['Remark 3']] = 'Sowmya Ravisandiran'

In [ ]:
ash_data.loc[ash_data['Year'] <= 2021,['Data Quality']] = 2
ash_data.loc[ash_data['Year'] > 2035,['Data Quality']] = 4
ash_data.loc[(ash_data['Year'] > 2021)&(ash_data['Year']<=2035),['Data Quality']] = 3

In [ ]:
ash_data = ash_data[columns]
#eu_msw_his.drop(eu_msw_his.loc[eu_msw_his["Country"]=='EU27_2020'].index, inplace = True)

In [ ]:
#ash_data.to_csv('Data_Structure_Task4.1_Task4.2_all_scenarios.csv', index = False)